# Proyecto LSP — Exportar el modelo a TensorFlow.js (para la demo web)

Convierte `modelo_pucp305_final.keras` → formato **TensorFlow.js** (`model.json` + pesos)
para que la demo HTML (`web/lsp_demo.html`) corra el modelo en el navegador, sin Python.

Genera un `web_model.zip` que descargas y descomprimes junto al HTML.

> **Nota:** el modelo es un BiLSTM. La conversión a TF.js suele funcionar, pero hay un bug
> conocido con capas Bidirectional. Si la **demo web** falla al cargar el modelo, avísame y
> usamos el plan B (servir el modelo desde un mini-backend Flask, sin convertir).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# tensorflowjs instala sus propias deps, INCLUIDA jax (su converter la importa).
# OJO: a diferencia de los notebooks de inferencia (04/05/06-setup), aquí NO se
# desinstala jax — hacerlo rompe `import tensorflowjs` (ModuleNotFoundError: jax).
!pip install -q tensorflowjs

In [ ]:
import shutil, json
from pathlib import Path
import tensorflow as tf
import tensorflowjs as tfjs

M = Path('/content/drive/MyDrive/PUCP305_models')
OUT = Path('/content/web_model'); OUT.mkdir(exist_ok=True)

model = tf.keras.models.load_model(M / 'modelo_pucp305_final.keras', compile=False)
print('Modelo cargado:', model.input_shape, '->', model.output_shape)

# Keras -> TensorFlow.js (LayersModel, lo carga el HTML con tf.loadLayersModel)
try:
    tfjs.converters.save_keras_model(model, str(OUT))
    print('Conversion OK.')
except Exception as e:
    print('FALLO la conversion a TF.js:', type(e).__name__, e)
    print('\nSi el error menciona Bidirectional / "no target variable", es el bug '
          'conocido del BiLSTM en TF.js -> avisame para el plan B (backend Flask).')
    raise

# clases junto al modelo (las usa el HTML)
shutil.copy(M / 'classes_pucp305.json', OUT / 'classes_pucp305.json')
print('\nArchivos generados:')
for p in sorted(OUT.iterdir()):
    print(f'  {p.name:30} {p.stat().st_size/1024:8.1f} KB')

In [ ]:
# Empaquetar y descargar
shutil.make_archive('/content/web_model', 'zip', OUT)
from google.colab import files
files.download('/content/web_model.zip')
print('Descomprime web_model.zip dentro de la carpeta web/ del repo (junto a lsp_demo.html).')